**This notebook is used for making recommendations for a given customer**

It only makes recommendations for items not already bought by the customer and uses the trained NCF model to justify its recommendations.

In [1]:
import pandas as pd

In [2]:
interactions = pd.read_csv('data_processed/interactions.csv')

In [3]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(interactions, test_size=0.2, random_state=42, shuffle=True)

In [4]:
# All items a user bought in TRAINING set
train_user_items = train.groupby('user_id')['item_id'].apply(set).to_dict()

# All items a user bought in TEST set  
test_user_items = test.groupby('user_id')['item_id'].apply(set).to_dict()

In [5]:
import torch
def get_user_inputs(user_id, num_items, interactions, device):
    """Prepares and returns the user, item, and cluster tensors on the correct device."""
    user_tensor = torch.tensor([user_id] * num_items, dtype=torch.long).to(device)
    item_tensor = torch.tensor(range(num_items), dtype=torch.long).to(device)
    
    # Extract cluster ID (safely handles the single row extraction)
    cluster_id = interactions[interactions["user_id"] == user_id].iloc[[0]]['cluster_id'].values[0]
    cluster_tensor = torch.tensor([cluster_id] * num_items, dtype=torch.long).to(device)
    
    return user_tensor, item_tensor, cluster_tensor


def get_top_k_recommendations(user_id, num_items, interactions, train_user_items, test_user_items, model, device, top_k=10):
    """Runs prediction for a single user and checks if any test items are in the top-K."""
    # 1. Get tensors
    user_tensor, item_tensor, cluster_tensor = get_user_inputs(user_id, num_items, interactions, device)
    
    # 2. Model Inference
    predictions = model(user_tensor, item_tensor, cluster_tensor).squeeze()

    # 3. Mask known train items
    known_items = list(train_user_items.get(user_id, set()))
    predictions[known_items] = float('-inf')

    # 4. Get Top-K recommendations
    scores, idx = torch.topk(predictions, top_k)
    return scores, idx



In [14]:
test

,user_id,item_id,purchased,cluster_id
1271749,2381,1834,0,0
1764808,3915,3893,0,0
1733785,3816,4549,0,3
1463271,2989,2730,0,2
118151,1510,1571,1,2
...,...,...,...,...
565689,266,3299,0,3
1879908,4297,3704,0,3
323183,3950,1427,1,0
1867972,4254,3593,0,1


In [17]:
# Filter for the specific column value, then sample 10 rows
customers_ids = test[test['cluster_id'] == 3].sample(n=10)['user_id'].copy()

print(customers_ids)
num_users = train['user_id'].nunique()
num_items = train['item_id'].nunique()
print(num_users, num_items)

1433559    2894
860532     1203
1158281    2124
246314     3005
341528     4169
274110     3349
181179     2224
1074292    1860
682183      652
964425     1546
Name: user_id, dtype: int64
5878 4631


In [18]:
import torch
from model import NCF
model = NCF(num_users, num_items, num_clusters=4)
model.load_state_dict(torch.load('model_weights/ncf_model.pth'))
model.eval()

NCF(
  (user_embedding): Embedding(5878, 64)
  (item_embedding): Embedding(4631, 64)
  (cluster_embedding): Embedding(4, 8)
  (fc1): Linear(in_features=136, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=16, bias=True)
  (output): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
  (sigmoid): Sigmoid()
)

In [9]:
df = pd.read_csv('data_processed/cleaned_retail_data.csv')

In [10]:
def item_id_to_stock_code(item_id, df):
    """Converts a user_id back to the original customer_id using the interactions DataFrame."""
    stock_code = df[df["item_id"] == item_id].iloc[0]['StockCode']
    return stock_code

In [11]:
def get_description(item_id, df):
    return df[df["item_id"] == item_id].iloc[0]['Description']



In [20]:
from config import NUM_ITEMS, NUM_USERS
def recommend_for_user(user_id,interactions, train_user_items, test_user_items, model, df, device, num_items=NUM_ITEMS, top_k=10):
    # 1. Get top k item ids
    _, rec_ids = get_top_k_recommendations(user_id, num_items=num_items, interactions=interactions, train_user_items=train_user_items, test_user_items=test_user_items, model=model, device=device, top_k=top_k)

    # 2. For each item_id get description
    recommendations = []
    for item_id in rec_ids:
        description = get_description(item_id.item(), df)
        recommendations.append(description)
    
    return recommendations

In [23]:
for customer in customers_ids:
    recommendation = recommend_for_user(customer, 
                                        interactions, 
                                        train_user_items, 
                                        test_user_items, 
                                        model, df, 
                                        device='cpu', 
                                        num_items=num_items, 
                                        top_k=10)
    print(f"Recommendations for user {customer}:")
    print(recommendation,'\n')


Recommendations for user 2894:
['WICKER STAR ', 'RED SPOTTY COIR DOORMAT', 'COLOUR GLASS T-LIGHT HOLDER HANGING', 'REGENCY CAKESTAND 3 TIER', 'BAKING SET 9 PIECE RETROSPOT ', 'WOOD BLACK BOARD ANT WHITE FINISH', 'JUMBO BAG RED WHITE SPOTTY ', 'ASSORTED COLOUR BIRD ORNAMENT', 'HEART OF WICKER SMALL', 'HOMEMADE JAM SCENTED CANDLES'] 

Recommendations for user 1203:
['WOODLAND CHARLOTTE BAG', 'JUMBO BAG OWLS', 'HOT WATER BOTTLE TEA AND SYMPATHY', 'RED SPOTTY CHARLOTTE BAG', 'HOME SWEET HOME METAL SIGN ', 'LUNCH BAG WOODLAND', 'CHARLOTTE BAG , PINK/WHITE SPOTS', 'LUNCHBOX WITH CUTLERY RETROSPOT ', 'ROUND SNACK BOXES ,SET4, WOODLAND ', 'RETRO SPOT STORAGE JAR'] 

Recommendations for user 2124:
['GROW YOUR OWN BASIL IN ENAMEL MUG', 'REX CASH+CARRY JUMBO SHOPPER', 'RETRO SPOT TEA SET CERAMIC 11 PC ', 'Manual', 'WHITE SKULL HOT WATER BOTTLE ', "POPPY'S PLAYHOUSE LIVINGROOM ", 'AIRLINE BAG VINTAGE TOKYO 78', 'BAKING SET 9 PIECE RETROSPOT ', 'PARTY CONE CHRISTMAS DECORATION ', 'PICTURE DOMINOES'